# Разбор Train-Time Support Для `O` В Coarse

Цель:
- Восстановить тот же `train/test split`, что использует coarse benchmark.
- Проверить, не теряется ли true `O` и hottest `O` tail еще до inference.
- Сравнить hot `O/B` boundary support в `full/train/test`.
- Зафиксировать, это support issue или уже чисто model-side `O -> B` boundary problem.

In [1]:
# Setup: repo root, sys.path и display-настройки.
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import seaborn as sns
from IPython.display import display

def find_repo_root(start_path: Path) -> Path:
    current_path = start_path.resolve()
    while current_path != current_path.parent:
        if (current_path / 'src').exists() and (current_path / '.git').exists():
            return current_path
        current_path = current_path.parent
    raise RuntimeError('Could not locate repository root.')

REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 160)


In [2]:
# Импортируем review helpers после добавления src в sys.path.
from exohost.reporting.archive_research.coarse_o_train_support_review import (
    CoarseOTrainSupportReviewConfig,
    build_benchmark_alignment_frame,
    build_hot_ob_boundary_support_frame,
    build_hottest_true_o_preview_frame,
    build_true_o_physics_summary_frame,
    build_true_o_split_support_frame,
    build_true_o_stage_support_frame,
    build_true_o_teff_band_support_frame,
    load_coarse_o_train_support_review_bundle_from_env,
)


## План

1. Загружаем prepared coarse source и reconstruct deterministic split.
2. Проверяем, совпадает ли reconstructed split с benchmark artifact.
3. Смотрим поддержку true `O` в `full/train/test`.
4. Проверяем `evolution_stage` и temperature-band support для true `O`.
5. Сравниваем hot `O/B` boundary support в `full/train/test`.
6. Смотрим hottest true `O` preview с фактическим split membership.
7. В конце фиксируем, support issue это или уже model-side boundary issue.

In [3]:
# Конфигурация notebook.
BENCHMARK_RUN_DIR = REPO_ROOT / 'artifacts/benchmarks/gaia_id_coarse_classification_2026_03_28_171258_103400'
SUPPORT_CONFIG = CoarseOTrainSupportReviewConfig(hot_teff_min_k=10000.0)
SOURCE_LIMIT = None
PREVIEW_ROWS = 20
DOTENV_PATH = REPO_ROOT / '.env'

if not BENCHMARK_RUN_DIR.exists():
    raise FileNotFoundError(BENCHMARK_RUN_DIR)


In [4]:
# Загружаем train-support review bundle и сверяем split с benchmark artifact.
bundle = load_coarse_o_train_support_review_bundle_from_env(
    benchmark_run_dir=BENCHMARK_RUN_DIR,
    config=SUPPORT_CONFIG,
    source_limit=SOURCE_LIMIT,
    dotenv_path=str(DOTENV_PATH),
)

display(build_benchmark_alignment_frame(bundle))


,split_name,n_rows_reconstructed,n_rows_benchmark,is_match
0,full,32986,32986,True
1,train,23090,23090,True
2,test,9896,9896,True


In [5]:
# Сколько true `O` вообще есть в `full/train/test`, и как они выглядят по stage.
display(build_true_o_split_support_frame(bundle))
display(build_true_o_stage_support_frame(bundle))


,split_name,n_rows_split,n_rows_true_o,share_true_o_in_split,share_true_o_of_total_o
0,full,32986,3000,0.090948,1.0
1,train,23090,2100,0.090948,0.7
2,test,9896,900,0.090946,0.3


,split_name,evolution_stage,n_rows,share_within_split_true_o
0,full,evolved,3000,1.0
1,train,evolved,2100,1.0
2,test,evolved,900,1.0


In [6]:
# Temperature-band support и медианная физика true `O` между `full/train/test`.
display(build_true_o_teff_band_support_frame(bundle))
display(build_true_o_physics_summary_frame(bundle))


,split_name,teff_band,n_rows,share_within_split_true_o
0,full,>= 25000 K,3000,1.0
1,test,>= 25000 K,900,1.0
2,train,>= 25000 K,2100,1.0


,split_name,n_rows_true_o,median_teff_gspphot,median_logg_gspphot,median_bp_rp,median_radius_feature,median_radius_flame
0,full,3000,34604.6170,3.89625,3.385037,8.79655,<NA>
1,train,2100,34673.9105,3.89415,3.384874,8.79365,<NA>
2,test,900,34334.5900,3.90220,3.385149,8.81005,<NA>


In [7]:
# Поддержка hot `O/B` boundary source еще до inference.
display(build_hot_ob_boundary_support_frame(bundle))


,split_name,spec_class,n_rows,share_within_hot_boundary
0,full,B,3000,0.5
1,full,O,3000,0.5
2,train,B,2100,0.5
3,train,O,2100,0.5
4,test,B,900,0.5
5,test,O,900,0.5


In [8]:
# Самые горячие true `O` rows и их train/test membership.
display(build_hottest_true_o_preview_frame(bundle, top_n=PREVIEW_ROWS))


,source_id,spec_class,evolution_stage,teff_gspphot,logg_gspphot,mh_gspphot,bp_rp,parallax,parallax_over_error,ruwe,radius_feature,radius_flame,split_name
0,5875511794958679424,O,evolved,40099.760,3.9907,-0.9316,3.860992,-0.082412,-2.427745,1.021208,9.1584,NaN,train
1,2176988290924113024,O,evolved,39972.285,3.9665,-0.6839,3.194766,0.827989,31.515911,1.074749,9.6408,NaN,test
2,1833937841256275968,O,evolved,39957.715,3.8195,-0.9881,3.445809,0.464042,15.627385,1.137370,12.5259,NaN,train
3,6731230393071987968,O,evolved,39933.285,3.8935,-0.8503,2.461670,1.114709,68.213173,0.923137,10.8631,NaN,train
4,2060124636016432000,O,evolved,38912.816,3.9529,-0.5007,3.573503,0.202646,4.419514,2.144050,9.5534,NaN,test
5,4152182452588548480,O,evolved,38779.230,3.9453,-0.9732,3.739063,0.076013,1.884802,1.085625,9.2088,NaN,train
6,6018359864547564416,O,evolved,38008.500,3.8892,-0.9497,3.723447,0.061912,1.510521,1.114169,10.2759,NaN,train
7,4317576003848306432,O,evolved,37640.797,3.8150,-0.7152,2.598650,0.179537,12.843781,1.025066,12.5664,NaN,train
8,4268196391248648576,O,evolved,37580.918,3.7562,-0.4103,2.518966,0.605795,30.294306,0.830461,12.9870,NaN,train
9,1825971608951073408,O,evolved,37498.156,3.7353,-0.9960,3.764195,0.302450,11.419653,0.963281,13.0715,NaN,train


## Вывод

На этом шаге мы проверяем не качество предсказаний, а наличие поддержки для класса `O` в самом coarse training source.

Если reconstructed split совпадает с benchmark artifact, а true `O` и hottest `O` tail полноценно представлены в `train/test`,
то проблема `O -> B` уже не объясняется нехваткой support. В таком случае следующий корректный шаг —
разбирать feature separability и class-boundary между `O` и `B`, а не пытаться лечить проблему только количеством `O` строк.